# Stop Your AI Agent from Forgetting User Preferences: Key-Value Memory (Agent State)

A new user arrives. The agent knows nothing about them. They search flights, book
one, and ask for a recommendation. Whether the agent can answer *"based on what you
know about me"* after a restart depends on one decision: where memory lives.

One clarification first: Strands keeps the conversation history (`agent.messages`)
between calls on the same agent instance, so within a session even a memory-less
agent "remembers" — the booking is still in the transcript. The failure is that the
transcript is the only place the preference exists. Nothing structured is learned,
and the moment the process restarts (every new request in production), it is gone.

The simplest agent memory is a key-value store: facts under named keys, no
embeddings. Strands calls it [agent state](https://strandsagents.com/docs/user-guide/concepts/agents/state/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).
This notebook climbs its durability ladder one rung at a time:

| Test | Memory wiring | Survives |
|------|---------------|----------|
| 1 | none (transcript only) | turns in one process, nothing after a restart |
| 2 | `agent.state` | turns, as a structured profile (within the process) |
| 3 | + `FileSessionManager` | restarts (local disk) |
| 4 | + `S3SessionManager` | restarts in the cloud (Amazon S3, plain JSON) |

The tools call live APIs (real flight offers from the [Duffel](https://duffel.com)
sandbox, real climate from [Open-Meteo](https://open-meteo.com)). The tool functions
are defined in the cells below; only the API clients (`flights_api`, `weather_api`)
are imported, since they are plumbing.

Based on:
- [MemoryOS of AI Agent](https://arxiv.org/abs/2506.06326) (Kang et al., 2025)
- [Cognitive Memory in Large Language Models](https://arxiv.org/abs/2504.02441) (Shan et al., 2025)

## Install dependencies

Run once, or from a terminal: `uv venv && uv pip install -r requirements.txt`.

In [ ]:
%pip install -q -r requirements.txt

## Configure credentials

- `OPENAI_API_KEY`: the model (or switch to Amazon Bedrock in the model cell).
- `DUFFEL_API_KEY`: free flight-search sandbox token from [app.duffel.com](https://app.duffel.com) (More → Developers → Access tokens).
- `SESSIONS_BUCKET` (optional, Test 4 only): an S3 bucket you own, plus AWS credentials. Without it, Tests 1-3 still run.

The climate API (Open-Meteo) needs no key.

In [ ]:
import os

# python-dotenv loads the keys from a local .env, so credentials never live
# in the notebook.
from dotenv import load_dotenv
load_dotenv()

assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env (or switch to Bedrock in the model cell)'
assert os.getenv('DUFFEL_API_KEY'), 'Set DUFFEL_API_KEY in .env (free token at https://app.duffel.com)'

## The model and the conversation

One model object, reused by every agent — the only cell you touch to change
providers. The same three-turn conversation drives every test, so the only
variable across tests is where memory lives.

In [ ]:
# OTEL_SDK_DISABLED silences OpenTelemetry tracing noise in notebook output.
os.environ['OTEL_SDK_DISABLED'] = 'true'

# Using the OpenAI-compatible interface via the Strands SDK.
from strands.models.openai import OpenAIModel
MODEL = OpenAIModel(model_id='gpt-4o-mini')   # api_key read from OPENAI_API_KEY

# Amazon Bedrock instead (uses your AWS credentials, no OpenAI key):
# from strands.models import BedrockModel
# MODEL = BedrockModel(model_id='openai.gpt-oss-120b-1:0', region_name='us-west-2')

SYSTEM_PROMPT = (
    'You are a flight booking assistant for a returning traveler. '
    'Personalize recommendations using what you know about the user. '
    'Be concise, answer in 2-3 sentences maximum.'
)

TURN_1 = 'Find me flights from JFK to Paris CDG on 2026-09-15, business class.'
TURN_2 = 'Book the cheapest business option.'
TURN_3 = ('Now I need Paris CDG to Tokyo Haneda on 2026-09-22, '
          'what do you recommend based on what you know about me?')

---
## Test 1: No memory tools

The stateless tools search and book real Duffel offers but have no way to write
state: plain `@tool` functions with no `ToolContext`. They are defined here so
you can see there is no place for a preference to go. Only `flights_api` (the
Duffel client) is imported.

In [ ]:
import json
from strands import Agent, tool

# flights_api is the Duffel sandbox client (search_offers, get_offer). Imported
# because it is an API client, not the lesson.
import flights_api


@tool
def search_flights_stateless(origin: str, destination: str, departure_date: str,
                             cabin_class: str = "economy") -> str:
    """Search live flight offers for a route on a date.

    Args:
        origin: IATA code of departure, e.g. "JFK".
        destination: IATA code of arrival, e.g. "CDG".
        departure_date: ISO date, e.g. "2026-09-15".
        cabin_class: economy, premium_economy, business, or first.
    """
    offers = flights_api.search_offers(origin, destination, departure_date, cabin_class)
    return json.dumps(offers, indent=1)


@tool
def book_flight_stateless(offer_id: str) -> str:
    """Confirm a booking for a chosen offer. Learns nothing: no ToolContext.

    Args:
        offer_id: the Duffel offer id from a previous search, e.g. "off_0000B8...".
    """
    offer = flights_api.get_offer(offer_id)
    if offer is None:
        return f"Offer '{offer_id}' not found or expired. Search again."
    return json.dumps({
        "status": "CONFIRMED", "offer_id": offer_id,
        "price": offer["price"], "currency": offer["currency"],
        "route": " -> ".join(f"{sl['origin']}-{sl['destination']}" for sl in offer["slices"]),
    })


print('stateless tools defined')

Run the three turns, then the restart. Watch two things: turn 3 is personalized
within the session (the transcript still holds "business class"), but a new agent
instance — every process/request in production — gets turn 3 with an empty
transcript and nothing to go on. `user_preferences` stays `None` throughout.

In [ ]:
agent_stateless = Agent(
    model=MODEL, system_prompt=SYSTEM_PROMPT,
    tools=[search_flights_stateless, book_flight_stateless],
    callback_handler=None,
)

for turn in (TURN_1, TURN_2, TURN_3):
    resp = agent_stateless(turn)
    print(f'User: {turn}\nAgent: {str(resp).strip()[:220]}\n')

print('user_preferences after 3 turns:', agent_stateless.state.get('user_preferences'))
print(f'messages in transcript: {len(agent_stateless.messages)} (the booking lives ONLY here)')

# The restart: a NEW agent instance, no session manager, empty transcript.
agent_restarted = Agent(
    model=MODEL, system_prompt=SYSTEM_PROMPT,
    tools=[search_flights_stateless, book_flight_stateless],
    callback_handler=None,
)
resp = agent_restarted(TURN_3)
print(f'\n[after restart] User: {TURN_3}\n[after restart] Agent: {str(resp).strip()[:220]}')

---
## Test 2: `agent.state`, where the booking teaches the agent

Same conversation, but the tools carry `@tool(context=True)`: Strands injects a
`ToolContext`, and `tool_context.agent.state` is a key-value store outside the
conversation. `book_flight` writes what the choice reveals (cabin, stops, price
band, carriers); `search_flights` reads it back and ranks real offers by it in
code, with a deterministic `score()`. Both tools are defined here — this ranking
logic is the technique the demo teaches.

In [ ]:
from strands import ToolContext


@tool(context=True)
def search_flights(origin: str, destination: str, departure_date: str,
                   tool_context: ToolContext, cabin_class: str = "economy") -> str:
    """Search live offers, ranked by the user's learned profile when one exists.

    Args:
        origin: IATA code of departure, e.g. "JFK".
        destination: IATA code of arrival, e.g. "CDG".
        departure_date: ISO date, e.g. "2026-09-15".
        cabin_class: default cabin; a learned preference overrides it.
    """
    prefs = tool_context.agent.state.get("user_preferences") or {}
    effective_cabin = prefs.get("preferred_cabin") or cabin_class
    offers = flights_api.search_offers(origin, destination, departure_date, effective_cabin)

    if prefs:
        max_price = prefs.get("typical_price", {}).get("max")

        def score(offer):
            s = 0
            if all(sl["stops"] == 0 for sl in offer["slices"]) and prefs.get("prefers_nonstop"):
                s += 10
            if max_price and offer["price"] <= max_price:
                s += 5
            carriers = {seg["carrier"] for sl in offer["slices"] for seg in sl["segments"]}
            if carriers & set(prefs.get("carriers_flown", [])):
                s += 2
            return s

        offers.sort(key=score, reverse=True)
        header = f"Ranked by your preferences: {json.dumps(prefs)}\n\n"
    else:
        header = "No preferences learned yet. Book a flight to start building your profile.\n\n"
    return header + json.dumps(offers, indent=1)


@tool(context=True)
def book_flight(offer_id: str, tool_context: ToolContext) -> str:
    """Confirm a booking AND learn preferences from the chosen offer.

    The chosen offer's cabin, stops, price, and carriers are written to the
    profile so future searches rank by them.

    Args:
        offer_id: the Duffel offer id from a previous search, e.g. "off_0000B8...".
    """
    offer = flights_api.get_offer(offer_id)
    if offer is None:
        return f"Offer '{offer_id}' not found or expired. Search again."

    prefs = tool_context.agent.state.get("user_preferences") or {}
    prefs["preferred_cabin"] = offer["cabin"]
    prefs["prefers_nonstop"] = all(sl["stops"] == 0 for sl in offer["slices"])
    carriers = sorted({seg["carrier"] for sl in offer["slices"] for seg in sl["segments"] if seg["carrier"]})
    prefs["carriers_flown"] = sorted(set(prefs.get("carriers_flown", [])) | set(carriers))
    price_range = prefs.get("typical_price", {})
    prefs["typical_price"] = {
        "min": min(price_range.get("min", offer["price"]), offer["price"]),
        "max": max(price_range.get("max", offer["price"]), offer["price"]),
    }
    tool_context.agent.state.set("user_preferences", prefs)
    return json.dumps({"status": "CONFIRMED", "offer_id": offer_id, "preferences_updated": prefs})


@tool(context=True)
def get_user_profile(tool_context: ToolContext) -> str:
    """Show what the agent has learned about this user so far."""
    prefs = tool_context.agent.state.get("user_preferences")
    if not prefs:
        return "No user profile yet."
    return json.dumps({"preferences": prefs}, indent=1)


# best_time_to_visit uses the Open-Meteo client; user-independent, so imported.
import weather_api

@tool
def best_time_to_visit(city: str) -> str:
    """Answer "when should I visit X?" with real historical climate by month.

    Args:
        city: city name, e.g. "Tokyo".
    """
    climate = weather_api.monthly_climate(city)
    if climate is None:
        return f"Could not retrieve climate data for '{city}'."
    return json.dumps(climate, indent=1)


print('stateful tools defined')

In [ ]:
from strands.agent.conversation_manager import SlidingWindowConversationManager

agent_stateful = Agent(
    model=MODEL, system_prompt=SYSTEM_PROMPT,
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=[search_flights, book_flight, get_user_profile, best_time_to_visit],
    callback_handler=None,
)

for turn in (TURN_1, TURN_2, TURN_3):
    resp = agent_stateful(turn)
    print(f'User: {turn}\nAgent: {str(resp).strip()[:220]}\n')

print('user_preferences:', json.dumps(agent_stateful.state.get('user_preferences'), indent=1))

The difference from Test 1 is what exists afterwards: a structured, inspectable
profile in `agent.state`, built from the booking action. The search tool ranks
offers with it in code; `get_user_profile` can show it; a session manager can
persist it (next test). In Test 1 the same knowledge lived only as prose in the
transcript.

---
## Test 3: `FileSessionManager`, the profile survives a restart

`agent.state` lives in the process, so a restart wipes it, like the transcript in
Test 1. `FileSessionManager` persists state to disk keyed by `session_id`: a new
agent instance with the same id starts already knowing the user. (In production
you would use `S3SessionManager`, the same interface with S3 storage.)

In [ ]:
from strands.session import FileSessionManager
import shutil

session_id = 'traveler-demo'
storage_dir = os.path.join(os.path.abspath('.'), 'sessions')

def make_agent():
    return Agent(
        model=MODEL, system_prompt=SYSTEM_PROMPT,
        conversation_manager=SlidingWindowConversationManager(window_size=40),
        tools=[search_flights, book_flight, get_user_profile, best_time_to_visit],
        session_manager=FileSessionManager(session_id=session_id, storage_dir=storage_dir),
        callback_handler=None,
    )

# Session A: the user books, the profile is built.
agent_a = make_agent()
agent_a(TURN_1)
agent_a(TURN_2)
prefs_a = agent_a.state.get('user_preferences')
print('Session A learned:', json.dumps(prefs_a))

# Session B: NEW instance, same session_id (the restart).
agent_b = make_agent()
prefs_b = agent_b.state.get('user_preferences')
print('Session B restored:', json.dumps(prefs_b))
print('State survived restart:', prefs_a == prefs_b and prefs_b is not None)

resp = agent_b(TURN_3)
print(f'\nUser: {TURN_3}\nAgent: {str(resp).strip()[:250]}')

shutil.rmtree(storage_dir, ignore_errors=True)

---
## Test 4: `S3SessionManager`, the same rung in the cloud

`S3SessionManager` is the same interface as `FileSessionManager`, but the session
persists as plain JSON objects in Amazon S3 (no vectors). When to choose it over
disk: nothing to provision or mount (a durable filesystem on Lambda/Fargate means
wiring up EFS), and any compute instance can restore the session. Same test: a new
agent instance with the same `session_id` restores the profile from a bucket.

In [ ]:
from strands.session import S3SessionManager
import boto3

bucket = os.getenv('SESSIONS_BUCKET')
assert bucket, 'Set SESSIONS_BUCKET in .env to run this test (created if it does not exist)'

# Self-provisioning: create the bucket if missing (private, public access blocked).
# us-east-1 must NOT send a LocationConstraint; other regions require it.
s3 = boto3.client('s3')
try:
    s3.head_bucket(Bucket=bucket)
except s3.exceptions.ClientError:
    region = s3.meta.region_name or 'us-east-1'
    params = {'Bucket': bucket}
    if region != 'us-east-1':
        params['CreateBucketConfiguration'] = {'LocationConstraint': region}
    s3.create_bucket(**params)
    s3.put_public_access_block(Bucket=bucket, PublicAccessBlockConfiguration={
        'BlockPublicAcls': True, 'IgnorePublicAcls': True,
        'BlockPublicPolicy': True, 'RestrictPublicBuckets': True})
    print(f'created private bucket s3://{bucket} in {region}')

s3_session_id = 'traveler-demo-s3'
prefix = 'kv-memory-demo'

def make_s3_agent():
    return Agent(
        model=MODEL, system_prompt=SYSTEM_PROMPT,
        conversation_manager=SlidingWindowConversationManager(window_size=40),
        tools=[search_flights, book_flight, get_user_profile, best_time_to_visit],
        session_manager=S3SessionManager(session_id=s3_session_id, bucket=bucket, prefix=prefix),
        callback_handler=None,
    )

agent_a = make_s3_agent()
agent_a(TURN_1)
agent_a(TURN_2)
prefs_a = agent_a.state.get('user_preferences')
print('Session A learned:', json.dumps(prefs_a))

agent_b = make_s3_agent()   # new instance, same session_id, restored FROM S3
prefs_b = agent_b.state.get('user_preferences')
print(f'Session B restored from s3://{bucket}/{prefix}:', json.dumps(prefs_b))
print('State survived restart:', prefs_a == prefs_b and prefs_b is not None)

# Clean up this demo's session objects so reruns start empty.
listed = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
keys = [{'Key': o['Key']} for o in listed.get('Contents', [])]
if keys:
    s3.delete_objects(Bucket=bucket, Delete={'Objects': keys})

---
### Deterministic vs model-based here

The control lives in the agent's harness. Memory is [`agent.state`](https://strandsagents.com/docs/user-guide/concepts/agents/state/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)
plus a session manager, both native to the agent.

Deterministic (plain code): writing the profile to `agent.state`, ranking offers with
`score()`, and persisting through `FileSessionManager` / `S3SessionManager`. Same input,
same result. Model-based: the agent choosing which tool to call and wording the reply.
A model call carries no reproducibility guarantee across runs (neural-network inference
varies with floating-point non-associativity even under greedy decoding,
[Enabling Determinism in LLM Inference](https://arxiv.org/abs/2601.17768), 2026). What
the demo measures, whether a structured profile exists and survives a restart, is the
deterministic part.

---
## Summary

| Test | Memory wiring | Structured profile | Survives restart |
|------|---------------|--------------------|------------------|
| 1 | none (transcript only) | No | No |
| 2 | `agent.state` | Yes | No |
| 3 | + `FileSessionManager` | Yes | Yes (local disk) |
| 4 | + `S3SessionManager` | Yes | Yes (Amazon S3) |

Memory is wiring, not model. Test 1 looked fine within the session (the transcript
carried the preference) but learned nothing structured, and one restart erased it.
Tests 2-4 made the same knowledge explicit (`agent.state`) and then durable, one
rung at a time: process → disk → S3.

Next: [Demo 02: Vector Memory](../02-vector-memory-demo/) puts the JSON into a
vector store and retrieves by meaning.

---
## Cleanup: delete AWS resources (optional)

Run this only to remove the S3 bucket created by Test 4. Tests 1-3 create no AWS
resources. If you skip it, the bucket persists (it holds small JSON session
objects, costing fractions of a cent).

In [ ]:
import boto3, os
from dotenv import load_dotenv
load_dotenv()

bucket = os.getenv('SESSIONS_BUCKET')
if not bucket:
    print("SESSIONS_BUCKET not set, nothing to delete.")
else:
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    objects = [{'Key': obj['Key']}
               for page in paginator.paginate(Bucket=bucket)
               for obj in page.get('Contents', [])]
    if objects:
        s3.delete_objects(Bucket=bucket, Delete={'Objects': objects})
        print(f"  deleted {len(objects)} object(s) from s3://{bucket}")
    s3.delete_bucket(Bucket=bucket)
    print(f"  deleted bucket s3://{bucket}")